In [50]:
import json
import os
from typing import Dict, Any, List, Optional
import pandas as pd
from IPython.display import display

import plotly.express as px
import plotly.graph_objects as go

default_sample = """\
{"timestamp": 1759790081.822003, "device": "phone_2", "action": "idle", "time": 0.01679604311979629}
{"timestamp": 1759790081.822016, "time": 0.0799571066350519, "action": "transmit_end", "id": "all_gather_reduce_from_model_parallel_region_phone_1_phone_2_0", "internal_id": 1363, "duration": 0.07533652252252253}
{"timestamp": 1759790081.822017, "device": "phone_2", "action": "running", "time": 0.0799571066350519}
{"timestamp": 1759790081.846875, "time": 0.10479856463331555, "action": "transmit_start", "id": "all_gather_reduce_from_model_parallel_region_phone_2_phone_1_0", "internal_id": 1364, "size": 8286208.0}
{"timestamp": 1759790081.846889, "device": "phone_2", "action": "idle", "time": 0.1048207316385425}
{"timestamp": 1759790081.84691, "time": 0.09211773163821374, "action": "transmit_end", "id": "all_gather_reduce_from_model_parallel_region_phone_2_phone_1_0", "internal_id": 1362, "duration": 0.07533652252252253}
{"timestamp": 1759790081.846913, "device": "phone_1", "action": "running", "time": 0.09211773163821374}
{"timestamp": 1759790081.89344, "time": 0.13863189764794254, "action": "transmit_start", "id": "all_gather_reduce_from_model_parallel_region_phone_1_phone_2_0", "internal_id": 1365, "size": 8286208.0}
{"timestamp": 1759790081.8934531, "device": "phone_1", "action": "idle", "time": 0.1386457726371783}
{"timestamp": 1759790081.893468, "device": "phone_2", "action": "running", "time": 0.1048207316385425}
"""


def parse_src_dst(name: str) -> tuple[Optional[str], Optional[str]]:
    if not name:
        return None, None
    parts = name.split("_")
    try:
        idx = None
        for i in range(len(parts) - 1, -1, -1):
            if parts[i].isdigit():
                idx = i
                break
        if idx is None:
            return None, None

        def pop_name_num(j):
            if j - 1 >= 0 and parts[j - 1].isdigit() and j - 2 >= 0:
                return f"{parts[j - 2]}_{parts[j - 1]}", j - 2
            return parts[j - 1], j - 1

        dst, j = pop_name_num(idx)
        src, _ = pop_name_num(j)
        return src, dst
    except Exception:
        return None, None


def load_jsonl_or_sample(path: str) -> List[Dict[str, Any]]:
    events = []
    if os.path.exists(path):
        with open(path, "r") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    events.append(json.loads(line))
                except json.JSONDecodeError:
                    continue
    if not events:
        events = [json.loads(line) for line in default_sample.strip().splitlines()]
    return events


events = load_jsonl_or_sample("../profile_out/event_log.jsonl")


starts: Dict[int, Dict[str, Any]] = {}
ends: Dict[int, Dict[str, Any]] = {}
meta: Dict[int, Dict[str, Any]] = {}

for ev in events:
    if not isinstance(ev, dict):
        continue
    action = ev.get("action", "")
    if not action.startswith("transmit_"):
        continue

    iid = ev.get("internal_id")
    if iid is None:
        continue

    if iid not in meta:
        meta[iid] = {}
    for k in ("id", "size"):
        if k in ev and k not in meta[iid]:
            meta[iid][k] = ev[k]

    if action == "transmit_start":
        starts[iid] = ev
    elif action == "transmit_end":
        ends[iid] = ev

rows = []
for iid, m in meta.items():
    s = starts.get(iid)
    e = ends.get(iid)

    name = m.get("id", f"internal_{iid}")
    size = m.get("size")
    src, dst = parse_src_dst(name)

    start_t = None
    end_t = None
    duration = None

    if s and "time" in s:
        start_t = float(s["time"])
    if e and "time" in e:
        end_t = float(e["time"])

    if s and "duration" in s:
        duration = float(s["duration"])
    if e and "duration" in e:
        duration = float(e["duration"])

    if start_t is not None and end_t is None and duration is not None:
        end_t = start_t + duration
    if end_t is not None and start_t is None and duration is not None:
        start_t = end_t - duration

    if (start_t is None or end_t is None) and e and ("time" in e) and ("duration" in e):
        end_t = float(e["time"])
        start_t = end_t - float(e["duration"])

    if start_t is None or end_t is None:
        continue

    rows.append(
        {
            "internal_id": iid,
            "name": name,
            "start": start_t,
            "end": end_t,
            "duration_s": end_t - start_t,
            "size_bytes": float(size) if size is not None else None,
            "src": src,
            "dst": dst,
        }
    )

df = pd.DataFrame(rows).sort_values(by=["start", "end", "internal_id"]).reset_index(drop=True)
df["internal_id_str"] = df["internal_id"].astype(str)
df["start"] = pd.to_numeric(df["start"], errors="coerce")
df["end"] = pd.to_numeric(df["end"], errors="coerce")

if df.empty:
    print("No intervals could be built from transmit events (need transmit_end with duration, or start+end pairs).")
else:
    fig = go.Figure()

    for name, dfg in df.groupby("name"):
        fig.add_bar(
            orientation="h",
            y=dfg["internal_id_str"],
            x=dfg["duration_s"],      # bar width = duration
            base=dfg["start"],        # bar start position
            name=name,
            hovertext=dfg.apply(
                lambda r: f"{r['name']}<br>ID:{r['internal_id']}<br>"
                        f"start:{r['start']:.6f}s <br>"
                        f"end:{r['end']:.6f}s<br>"
                        f"dur:{r['duration_s']:.6f}s <br>"
                        f"[src:{r['src']}] - [dst:{r['dst']}]<br>"
                        f"size: {r['size_bytes'] / 1024:.2f} KB" if r['size_bytes'] is not None else "size: N/A",
                axis=1,
            ),
            hoverinfo="text",
        )

    fig.update_yaxes(autorange="reversed", title="Internal ID")
    fig.update_xaxes(title="Simulation Time (s)", type="linear", rangeslider_visible=False)
    fig.update_layout(barmode="overlay", hovermode="closest", legend_title_text="Role (id)")
    display(fig)